# Docker execution policies

`DockerExecutionPolicy` runs tool commands in a separate Linux container. The agent and model inference stay on the host. An execution policy controls the container image, networking, resource limits, user identity and command deadline.

[Container execution](https://docs.langchain.com/oss/python/langchain/middleware/built-in#shell-tool)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.config import settings
from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.faults import scripted_model
from lab.sandbox import docker_ready, managed_docker_policy, run_worker
from langchain.agents import create_agent
from langchain.agents.middleware import ShellToolMiddleware
from model_config import get_model, run_config

In [ ]:
print("Container limits:", settings()["sandbox"])
docker_available = docker_ready()
if not docker_available:
    print("SKIPPED: Docker examples require the Linux engine and configured image.")

## Containerized workers

Each worker runs in its own named container with a read-only fixture mount, no network, a non-root user and CPU, memory and process limits. The calculation returns `42`; the nonterminating workers reach the deadline. Tool isolation applies to the container, while model requests happen outside it.

In [ ]:
if docker_available:
    docker_hello = run_worker("hello", mode="docker")
    print(docker_hello)
    docker_wait_forever = run_worker("wait_forever", mode="docker")
    print(docker_wait_forever)
    docker_spin_forever = run_worker("spin_forever", mode="docker")
    print(docker_spin_forever)
    assert docker_hello["status"] == "completed" and "42" in docker_hello["stdout"]
    assert docker_wait_forever["status"] == docker_spin_forever["status"] == "timed_out"

## A persistent shell inside the container

The agent selects commands through the `shell` tool. Commands within the invocation share one shell session. Its workspace is ephemeral, no host directory is mounted, and no notebook credentials are passed into the tool environment.

In [ ]:
# Execute
show_source(managed_docker_policy)
if docker_available:
    with managed_docker_policy() as policy:
        docker_shell = ShellToolMiddleware(execution_policy=policy, env={})
        agent = create_agent(model=get_model(), middleware=[docker_shell, TraceMiddleware("10_docker_shell", pretty=True)])
        shell_result = agent.invoke({"messages": [("user", prompt("shell_hello"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
if docker_available:
    show_messages(shell_result["messages"], pretty=True)

## Deadline and resource cleanup

An infinite shell command exceeds the deadline and produces a tool error. The policy helper removes the exact named container in `finally` and verifies it disappeared. Killing the Docker CLI alone does not guarantee that the container stopped.

In [ ]:
# Execute
if docker_available:
    with managed_docker_policy() as policy:
        agent = create_agent(
            model=scripted_model("shell_timeout"),
            middleware=[
                ShellToolMiddleware(execution_policy=policy, env={}),
                TraceMiddleware("10_timeout", pretty=True),
            ],
        )
        timeout_result = agent.invoke({"messages": [("user", prompt("shell_loop"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
if docker_available:
    show_messages(timeout_result["messages"], pretty=True)

## Isolation limits

Container isolation depends on the daemon and host configuration. A deadline limits time; memory, CPU and process limits constrain different resources. Cleanup targets only the invocation's container. A process kill or machine failure can prevent a Python `finally` block from running.